### Caiman "ground-truth" data

Available at https://zenodo.org/records/1659149

Download and unzip `images_J115.zip`, `images_K53.zip`, `images_YST.zip` and `WEBSITE.zip`


In [ ]:
import numpy as np 
from mat73 import loadmat
from pathlib import Path
from suite2p import default_settings, default_db, run_s2p, extraction
from suite2p.run_s2p import logger_setup
from suite2p.io import BinaryFile
from tifffile import imwrite, imread
from natsort import natsorted
from tqdm import tqdm, trange
from scipy.stats import zscore
import h5py
from cellpose import utils
import torch
logger_setup()

device = torch.device('cuda')

dsets = ['J115', 'K53', 'YST']

root0 = Path('/media/carsen/disk1/suite2p_paper/other_datasets/')

def detect_f1_iou(stat, stat_gt, Ly=512, Lx=512, iou_threshold=0.5):
    """ f1 score and matches """
    # find overlapping ROIs
    matched = np.zeros((len(stat_gt), len(stat)), 'float32')
    iou = np.zeros((len(stat_gt), len(stat)), 'float32')
    ly = 20
    for i in trange(len(stat)):
        sf = stat[i]
        if sf['ypix'].size < 10:
            continue
        ypix, xpix, lam = sf['ypix'].copy(), sf['xpix'].copy(), sf['lam'].copy()
        lam /= (lam**2).sum()**0.5
        # box around ROI
        if 'med' not in sf:
            ymed, xmed = int(np.median(ypix)), int(np.median(xpix))
        else:
            ymed, xmed = int(sf['med'][0]), int(sf['med'][1])
        inds = (slice(max(0, ymed - ly), min(ymed + ly, Ly)), slice(max(0, xmed - ly), min(xmed + ly, Lx)))
        mf = np.zeros((Ly, Lx), np.float32)
        mf[ypix, xpix] = lam
        mfc = mf > 0. *  lam.max()
        mfc = mfc[inds].flatten()
        
        # matched anatomical masks (will not compute IOU for all masks)
        for j, sa in enumerate(stat_gt):
            ypix_a, xpix_a = sa['ypix'], sa['xpix']
            if (np.logical_and(ypix_a > inds[0].start, ypix_a < inds[0].stop).sum() > 0
                    and np.logical_and(xpix_a > inds[1].start, xpix_a
                                        < inds[1].stop).sum() > 0):
                lam_a = sa['lam'].copy()
                ma = np.zeros((Ly, Lx), 'bool')
                ma[ypix_a, xpix_a] = lam_a > 0
                mac = ma[inds].flatten()
                intersection = (mac[mfc] > 0).sum()
                matched[j, i] = (intersection / mac.sum() > 0.5) * (intersection / mfc.sum() > 0.5)
                iou[j, i] = intersection / (mac.sum() + mfc.sum() - intersection)

    iou_filt = iou * (iou > iou_threshold)
    imatch_gt = iou_filt.max(axis=1) > 0. 
    imatch_uq = np.unique(iou_filt[imatch_gt].argmax(axis=1))
    print(len(imatch_uq), imatch_gt.sum())

    tp_comp = imatch_uq
    tp_gt = iou_filt[:,imatch_uq].argmax(axis=0)

    tp = len(imatch_uq)
    fp = len(stat) - tp
    fn = len(stat_gt) - tp
    f1 = tp / (tp + 0.5 * (fp + fn))

    print('TP: %d, FP: %d, FN: %d, F1: %.3f' % (tp, fp, fn, f1), flush=True)

    return f1, tp, fp, fn, iou, tp_gt, tp_comp

### Run suite2p

In [ ]:

for dset in dsets:
    root = Path(root0 / f'images_{dset}/')

    ### convert to multipage tiffs to run faster
    (root.parent / f'images_{dset}_multipage').mkdir(exist_ok=True)
    tifs = natsorted(list(root.glob('*.tif')))
    im0 = imread(tifs[0])
    Ly, Lx = im0.shape
    nframes = len(tifs)
    for i, tif in tqdm(enumerate(tifs)):
        if i % 2000 == 0:
            ims = np.zeros((min(2000, nframes - i), Ly, Lx), 'int16')
        im0 = imread(tif)
        if dset != 'YST':
            ims[i % 2000] = (im0 * 3).astype('int16')
            if (im0.max() * 3) > 2**15-1:
                print(f'AHH {i}')
        else:
            ims[i % 2000] = im0.astype('int16')

        if (i % 2000 == 1999) or (i == nframes - 1):
            imwrite(root.parent / f'images_{dset}_multipage' / f'ims_{i//2000:03d}.tif', ims)
    
    import shutil
    ### run suite2p
    rec = root0 / f'images_{dset}_multipage'
    shutil.rmtree(rec / 'suite2p', ignore_errors=True)
    db = default_db()
    db['data_path'] = [str(rec)]
    db['save_path0'] = str(rec)
    db['delete_bin'] = False
    db['nplanes'] = 1
    db['tau'] = 0.4
    db['fs'] = 30.0 if dset != 'YST' else 10.0
    settings = default_settings()

    ### run sparsery
    # change highpass from 25 to 40 because these are more zoomed in than our datasets
    settings['detection']['sparsery_settings']['highpass_neuropil'] = 40 if dset != 'YST' else 25
    run_s2p(settings=settings, db=db)

    from suite2p.run_s2p import run_s2p
    import shutil

    ### run sourcery
    # (small edit in suite2p run_s2p to make this work - need to fix)
    db['save_folder'] = 'suite2p_sourcery/'
    shutil.copytree(rec / 'suite2p', rec / 'suite2p_sourcery', dirs_exist_ok=True)
    
    settings = default_settings()
    settings['diameter'] = 16 if dset != 'YST' else 8
    settings['detection']['algorithm'] = 'sourcery'
    settings['detection']['threshold_scaling'] = 0.3
    settings['detection']['max_overlap'] = 0.85

    run_s2p(settings=settings, db=db)


    

### Benchmark using annotations

In [ ]:
results = {}

for dset in dsets:

    root = Path(root0 / f'images_{dset}_multipage/')
    db = np.load(root / 'suite2p/plane0/db.npy', allow_pickle=True).item()
    Ly, Lx = db['Ly'], db['Lx']

    ### GROUND-TRUTH ANNOTATIONS
    dat = np.load(root.parent / f'WEBSITE/{dset}/gt_eval.npz', allow_pickle=True)
    stat_gt = []
    A_gt = dat['A_gt'].item()
    ### YPIX AND XPIX ARE TRANSPOSED IN GT ###
    for i in trange(A_gt.shape[1]):
        if i in dat['idx_components_gt']:
            ipix = A_gt.row[A_gt.col == i]
            xpix, ypix = np.unravel_index(ipix, (Lx, Ly))
            lam = A_gt.data[A_gt.col == i]                    
            lam /= lam.sum()

            stat_gt.append({'ypix': ypix, 'xpix': xpix, 'lam': lam, 'ipix': ipix,
                        'radius': (len(ypix) / np.pi) ** 0.5 / 2,
                        'med': np.array([int(np.median(ypix)), int(np.median(xpix))]),
                        'overlap': np.zeros(len(ypix), 'bool'),
                        'npix': len(ypix)})
    stat_gt = np.array(stat_gt)

    ### SUITE2P results
    for alg in ['', '_sourcery']:
        cellprob = 0.35 if alg == '' else 0.25
        iscell = np.load(root / f'suite2p{alg}/plane0/iscell.npy')
        stat = np.load(root / f'suite2p{alg}/plane0/stat.npy', allow_pickle=True)
        npix_norm = np.array([s['npix_norm'] for s in stat])
        npix = np.array([s['npix'] for s in stat])
        npix_gt = np.array([s['npix'] for s in stat_gt])
        icell = (iscell[:,1] > cellprob) # set to best value for ~ all datasets to maximize F1 score
        npix_norm_min = 0.4 # set to best value for ~ all datasets to maximize F1 score
        icell *= (npix_norm > npix_norm_min)
        print(icell.sum())
        if alg == '':
            stat_s2p = stat[icell].copy()
        else:
            stat_s2p_sourcery = stat[icell].copy()

    ### CAIMAN results
    dat = np.load(list((root.parent / f'WEBSITE/{dset}').glob('*_perf_web_gsig.npz'))[0], allow_pickle=True)
    dat = dat['all_results'].item()

    A = dat['A'].T
    
    ncells = len(A.indptr) - 1
    stat_caiman = []
    for i in trange(ncells):
        ipix = A.indices[A.indptr[i]:A.indptr[i+1]]
        lam = A.data[A.indptr[i]:A.indptr[i+1]]

        xpix, ypix = np.unravel_index(ipix, (Lx, Ly)) # transposed

        # lam threshold (same as other figures)
        lam_threshold = 0.1 
        ilam = lam > lam_threshold * lam.max()
        ypix, xpix, lam = ypix[ilam], xpix[ilam], lam[ilam]
        lam /= lam.sum()

        stat_caiman.append({'ypix': ypix, 'xpix': xpix, 'lam': lam,
                        'radius': (len(ypix) / np.pi) ** 0.5,
                        'med': np.array([int(np.median(ypix)), int(np.median(xpix))]),
                        'overlap': np.zeros(len(ypix), 'bool'),
                        'npix': len(ypix)})

    stat_caiman = np.array(stat_caiman)

    iou_thresholds = [0.2, 0.5] # np.arange(0.2, 0.55, 0.1)
    scores_s2ps = np.zeros((len(iou_thresholds), 4))
    scores_s2p_sourcerys = np.zeros((len(iou_thresholds), 4))
    scores_caimans = np.zeros((len(iou_thresholds), 4))
    
    for i, iou_threshold in enumerate(iou_thresholds):
        f1_s2p, tp, fp, fn, iou, tp_gt_s2p, tp_comp_s2p = detect_f1_iou(stat_s2p, stat_gt, Ly=Ly, Lx=Lx, 
                                                                        iou_threshold=iou_threshold)
        scores_s2ps[i] = np.array([f1_s2p, tp, fp, fn])

        f1_s2p_sourcery, tp, fp, fn, iou, tp_gt_s2p_sourcery, tp_comp_s2p_sourcery = detect_f1_iou(stat_s2p_sourcery, stat_gt, Ly=Ly, Lx=Lx,
                                                                        iou_threshold=iou_threshold)
        scores_s2p_sourcerys[i] = np.array([f1_s2p_sourcery, tp, fp, fn])

        f1_caiman, tp, fp, fn, iou, tp_gt_caiman, tp_comp_caiman = detect_f1_iou(stat_caiman, stat_gt, Ly=Ly, Lx=Lx, 
                                                                                    iou_threshold=iou_threshold)
        scores_caimans[i] = np.array([f1_caiman, tp, fp, fn])

    results[dset] = {'iou_thresholds': iou_thresholds, 'scores_s2ps': scores_s2ps, 'scores_s2p_sourcerys': scores_s2p_sourcerys, 'scores_caimans': scores_caimans,
                     'matches_s2p': (tp_gt_s2p, tp_comp_s2p), 'matches_s2p_sourcery': (tp_gt_s2p_sourcery, tp_comp_s2p_sourcery), 'matches_caiman': (tp_gt_caiman, tp_comp_caiman),
                   'stat_s2p': stat_s2p, 'stat_s2p_sourcery': stat_s2p_sourcery, 'stat_caiman': stat_caiman, 'stat_gt': stat_gt}

for i, dset in enumerate(dsets):
    root = Path(root0 / f'images_{dset}_multipage/')
    reg_outputs = np.load(root / 'suite2p/plane0/reg_outputs.npy', allow_pickle=True).item()
    db = np.load(root / 'suite2p/plane0/db.npy', allow_pickle=True).item()
    Ly, Lx = db['Ly'], db['Lx']
    yrange, xrange = reg_outputs['yrange'], reg_outputs['xrange']
    detect_outputs = np.load(root / 'suite2p/plane0/detect_outputs.npy', allow_pickle=True).item()
    max_proj = np.zeros((Ly, Lx), 'float32')
    max_proj[yrange[0]:yrange[1], xrange[0]:xrange[1]] = detect_outputs['max_proj']
    results[dset]['max_proj'] = max_proj

# save f1 scores and stats and max_proj
np.savez('results/caiman_gt_results.npz', results=results)

In [ ]:
results = np.load(root0 / 'caiman_gt_results.npz', allow_pickle=True)['results'].item()
for dset in dsets:
    print(dset)
    print(results[dset]['scores_s2ps'][:,0], results[dset]['scores_s2p_sourcerys'][:,0], results[dset]['scores_caimans'][:,0])

In [ ]:
import figures 
import importlib
importlib.reload(figures)

fig = figures.suppfig_gtcaiman(results)
fig.savefig('figures/suppfig_gtcaiman.pdf')